# Modul 1 - Praktikum D3

Notebook ini mengerjakan tugas individu Modul 1 bagian D3: channel warna, crop, channel Red, kotak acak, flip, rectangle/circle wajah, channel B, dan penulisan nama file pada citra.

## Jawaban Pertanyaan Praktikum D3

1. Gambar yang ditampilkan tanpa Matplotlib, misalnya dengan `cv2_imshow`, ditampilkan langsung sebagai image output. Dengan Matplotlib, gambar masuk ke sistem plot sehingga bisa diberi `figsize`, judul, sumbu, colormap, subplot, dan pengaturan visual lain. Jika image dari OpenCV belum dikonversi BGR ke RGB, warna pada Matplotlib dapat terlihat tertukar.

2. `int16` menyimpan bilangan bulat 16-bit, sedangkan `int32` menyimpan bilangan bulat 32-bit. `int32` punya rentang nilai lebih besar, tetapi memakai memori lebih banyak. Untuk citra 8-bit biasa, tipe `uint8` paling sesuai karena channel warna berada pada rentang 0-255.

3. `from google.colab.patches import cv2_imshow` digunakan di Google Colab sebagai pengganti `cv2.imshow()`. Fungsi `cv2.imshow()` membutuhkan jendela GUI lokal, sedangkan Colab berjalan di browser/server sehingga perlu `cv2_imshow()` agar gambar tampil di output notebook.

4. `from skimage import io` menyediakan fungsi input/output image, misalnya `io.imread()`, yang praktis untuk membaca image dari URL maupun path file.

## Import Library dan Fungsi Bantu

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from skimage import io
try:
    from google.colab import drive
    from google.colab.patches import cv2_imshow
except Exception:
    drive = None
    def cv2_imshow(image):
        if image.ndim == 3:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(6, 4))
        plt.imshow(image, cmap='gray' if image.ndim == 2 else None)
        plt.axis('off')
        plt.show()
plt.rcParams['figure.dpi'] = 110
np.random.seed(26)
def make_fallback_image(width=800, height=571):
    y, x = np.mgrid[0:height, 0:width]
    r = (x / width * 255).astype(np.uint8)
    g = (y / height * 255).astype(np.uint8)
    b = (128 + 80 * np.sin(x / 35) * np.cos(y / 45)).clip(0, 255).astype(np.uint8)
    return np.dstack([r, g, b])
def read_rgb_from_url(url):
    try:
        img = io.imread(url)
        if img.ndim == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        if img.shape[-1] == 4:
            img = img[:, :, :3]
        return img.astype(np.uint8)
    except Exception as exc:
        print(f'Gagal membaca {url}: {exc}. Menggunakan fallback image.')
        return make_fallback_image()
def show_rgb(image, title='', figsize=(6, 4)):
    plt.figure(figsize=figsize)
    plt.imshow(image)
    plt.title(title)
    plt.axis('off')
    plt.show()

## Koneksi Google Drive

In [ ]:
if drive is None:
    print('Mount Google Drive hanya bisa dijalankan di Google Colab.')
else:
    drive.mount('/content/drive')
    print('Google Drive terhubung di /content/drive')

## Data Image untuk Tugas

In [ ]:
unsplash_url = 'https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?auto=format&fit=crop&w=800&q=80'
img_rgb = read_rgb_from_url(unsplash_url)
show_rgb(img_rgb, 'Image sumber: Unsplash', figsize=(7, 5))
print('Shape image:', img_rgb.shape)

## Tugas 1 - Pengaruh `figsize` terhadap Ukuran Pixel

In [ ]:
print('Ukuran pixel sebelum ditampilkan:', img_rgb.shape)
plt.figure(figsize=(3, 2))
plt.imshow(img_rgb)
plt.title('figsize kecil')
plt.axis('off')
plt.show()
plt.figure(figsize=(10, 6))
plt.imshow(img_rgb)
plt.title('figsize besar')
plt.axis('off')
plt.show()
print('Ukuran pixel setelah ditampilkan:', img_rgb.shape)
print('Kesimpulan: figsize hanya mengubah ukuran tampilan plot, bukan jumlah pixel pada array image.')

## Tugas 2 - Channel Red-Blue dan Green-Blue

In [ ]:
red_blue = img_rgb.copy()
red_blue[:, :, 1] = 0
green_blue = img_rgb.copy()
green_blue[:, :, 0] = 0
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(red_blue)
axes[0].set_title('Channel Red-Blue')
axes[0].axis('off')
axes[1].imshow(green_blue)
axes[1].set_title('Channel Green-Blue')
axes[1].axis('off')
plt.show()

## Tugas 3 - Crop Baris 20-115 dan Kolom 25-120

In [ ]:
crop_img = img_rgb[20:116, 25:121]
show_rgb(crop_img, 'Crop baris 20-115, kolom 25-120', figsize=(4, 4))
print('Shape crop:', crop_img.shape)

## Tugas 4 - Baris 5-30, Semua Kolom, Channel Red

In [ ]:
red_slice = img_rgb[5:31, :, 0]
plt.figure(figsize=(10, 2.2))
plt.imshow(red_slice, cmap='Reds')
plt.title('Baris 5-30, semua kolom, channel Red')
plt.axis('off')
plt.show()
print('Shape red slice:', red_slice.shape)

## Tugas 5 - Lima Kotak Berbagai Ukuran dan Warna

In [ ]:
img_kotak = img_rgb.copy()
h, w = img_kotak.shape[:2]
for i in range(5):
    box_w = np.random.randint(45, max(46, w // 4))
    box_h = np.random.randint(35, max(36, h // 4))
    x1 = np.random.randint(0, w - box_w)
    y1 = np.random.randint(0, h - box_h)
    x2 = x1 + box_w
    y2 = y1 + box_h
    color = np.random.randint(0, 256, size=3).tolist()
    cv2.rectangle(img_kotak, (x1, y1), (x2, y2), color, thickness=-1)
    print(f'Kotak {i + 1}: ({x1}, {y1}) sampai ({x2}, {y2}), warna RGB={color}')
show_rgb(img_kotak, 'Lima kotak acak dengan ukuran dan warna berbeda', figsize=(8, 5))

## Tugas 6 - Image Posisi Terbalik

In [ ]:
img_terbalik = cv2.flip(img_rgb, 0)
show_rgb(img_terbalik, 'Image terbalik vertikal', figsize=(7, 5))

## Tugas 7 - Rectangle dan Circle pada Wajah Foto Aktivitas

Cell ini membaca foto aktivitas pribadi dari Google Drive, lalu memberi rectangle dan circle pada area wajah.

In [ ]:
foto_aktivitas_path = '/content/drive/MyDrive/PCVK/foto_aktivitas.jpg'  # Ganti sesuai Copy path dari Google Drive
foto_bgr = cv2.imread(foto_aktivitas_path)
if foto_bgr is None:
    raise FileNotFoundError(f'File tidak dapat dibaca. Cek path Google Drive: {foto_aktivitas_path}')
foto_rgb = cv2.cvtColor(foto_bgr, cv2.COLOR_BGR2RGB)
foto_wajah = foto_rgb.copy()
h, w = foto_wajah.shape[:2]
def titik(x_ratio, y_ratio):
    return int(x_ratio * w), int(y_ratio * h)
# Wajah kiri
cv2.rectangle(foto_wajah, titik(0.00, 0.40), titik(0.43, 0.68), (255, 0, 0), 8)
cv2.circle(foto_wajah, titik(0.22, 0.54), int(0.23 * w), (0, 255, 0), 8)
# Wajah kanan
cv2.rectangle(foto_wajah, titik(0.51, 0.36), titik(0.82, 0.58), (255, 0, 0), 8)
cv2.circle(foto_wajah, titik(0.665, 0.47), int(0.16 * w), (0, 255, 0), 8)
show_rgb(foto_wajah, 'Rectangle dan circle pada area wajah', figsize=(7, 5))

## Tugas 8 - Rectangle pada Sudut Bawah Kiri Channel B di Color Space RGB

In [ ]:
img_channel_b = np.zeros_like(img_rgb)
img_channel_b[:, :, 2] = img_rgb[:, :, 2]
h, w = img_channel_b.shape[:2]
margin = 24
rect_w, rect_h = 180, 110
pt1 = (margin, h - rect_h - margin)
pt2 = (margin + rect_w, h - margin)
cv2.rectangle(img_channel_b, pt1, pt2, (255, 255, 255), 4)
show_rgb(img_channel_b, 'Channel B RGB dengan rectangle sudut bawah kiri', figsize=(7, 5))

## Tugas 9 - Menulis Nama File pada Citra Tugas 8

In [ ]:
img_teks = img_channel_b.copy()
nama_file = 'unsplash_cat.jpg'
font = cv2.FONT_HERSHEY_SIMPLEX
font_scale = 1.1
thickness = 3
text_org = (margin, max(40, h - rect_h - 45))
cv2.putText(img_teks, nama_file, text_org, font, font_scale, (255, 255, 0), thickness, cv2.LINE_AA)
show_rgb(img_teks, 'Channel B dengan rectangle dan nama file', figsize=(7, 5))